In [14]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from linearmodels.panel import PanelOLS

In [15]:
data = pd.read_excel('통합_1.xlsx')
data

,날짜,지역,주택구입부담지수,기준금리,거래량,인구수,물가지수,주택인허가실적,실거래가격지수
0,2021-01-01,서울,163.970000,0.50,12275,9657969,100.86,2695,167.6
1,2021-02-01,서울,166.200000,0.50,12707,9648606,101.33,9599,169.9
2,2021-03-01,서울,168.430000,0.50,11122,9598484,101.57,16290,170.8
3,2021-04-01,서울,169.870000,0.50,11873,9588711,101.68,26397,171.9
4,2021-05-01,서울,172.900000,0.50,13145,9575355,101.74,30915,175.4
...,...,...,...,...,...,...,...,...,...
811,2024-08-01,제주,72.300000,3.50,532,671540,114.25,2028,109.4
812,2024-09-01,제주,73.827162,3.50,485,671064,114.09,2254,109.7
813,2024-10-01,제주,76.002123,3.25,614,670837,114.34,2456,111.4
814,2024-11-01,제주,75.600000,3.00,557,670632,114.11,2557,109.7


In [16]:
data['날짜'] = pd.to_datetime(data['날짜'])
data = data.set_index(['지역', '날짜'])

패널 회귀분석 고정모형

In [17]:
X = data[['주택구입부담지수', '기준금리', '거래량', '인구수', '물가지수', '주택인허가실적']]
y = data['실거래가격지수']
X = sm.add_constant(X)

In [18]:
data.index.names

FrozenList(['지역', '날짜'])

In [20]:
data_sorted = data.sort_index(level='날짜')
train_idx = data_sorted.index.get_level_values('날짜') <'2024-01-01'
test_idx = data_sorted.index.get_level_values('날짜') > '2024-01-01'

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [21]:
model = PanelOLS(y_train, X_train, entity_effects=True)
results = model.fit()

In [22]:
y_pred = results.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse

np.float64(12.261276537152701)

In [23]:
print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                실거래가격지수   R-squared:                        0.6736
Estimator:                   PanelOLS   R-squared (Between):              0.7154
No. Observations:                 612   R-squared (Within):               0.6736
Date:                Mon, Nov 03 2025   R-squared (Overall):              0.7062
Time:                        15:01:44   Log-likelihood                   -1929.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      203.93
Entities:                          13   P-value                           0.0000
Avg Obs:                       47.077   Distribution:                   F(6,593)
Min Obs:                       36.000                                           
Max Obs:                       48.000   F-statistic (robust):             203.93
                            

패널 회귀분석 랜덤모형

In [24]:
from linearmodels.panel import RandomEffects

In [26]:
model = RandomEffects(y_train, X_train)
results = model.fit()
y_pred = results.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse

np.float64(12.131772446836454)

In [27]:
results.summary

Dep. Variable:,실거래가격지수,R-squared:,0.6740
Estimator:,RandomEffects,R-squared (Between):,0.7202
No. Observations:,612,R-squared (Within):,0.6735
Date:,"Mon, Nov 03 2025",R-squared (Overall):,0.7101
Time:,15:08:01,Log-likelihood,-1933.4
Cov. Estimator:,Unadjusted,,
,,F-statistic:,208.47
Entities:,13,P-value,0.0000
Avg Obs:,47.077,Distribution:,"F(6,605)"
Min Obs:,36.000,,
Max Obs:,48.000,F-statistic (robust):,208.67


17